# Prithvi-EO-2.0 with a year-based split: silage vs grain corn at BARC

Run on the GPU workspace from `classifier/prithvi/` (Run All). It imports `prithvi_train.py` and `prithvi_embed.py`
from this folder and changes neither. For a first pass, set `QUICK = True` in section 2: it checks that every cell
runs, in a few minutes, and its numbers are not results.

The earlier Prithvi run (`prithvi_train.py`, results in `classifier/README.md`) did not beat the 5-feature
logistic baseline (87 vs 90 of 101 test field-years), and four things make that result hard to trust. This notebook
changes each of them and nothing else.

| Problem in the earlier run | What this notebook does |
|---|---|
| **Coarse timing.** Four frames, one of them the median of Aug 10 - Sep 5, so the silage cut is averaged away. | Eight two-week frames (`prithvi_embed.FRAMES[8]`), for the frozen and the fine-tuned model, compared with four. |
| **The split mixes years.** Test field-years share a year (weather, image dates, cloud gaps) with training ones. | Year-based splits only. Headline: **train on 2017-2020, test on the 44 official 2021-2023 labels.** |
| **Weak model choice.** 40 setups ranked on about 20 validation field-years. | 20 pre-declared candidates, ranked only on hold-out-one-year inside 2017-2020 (83 field-years). The test is scored once, for the chosen one. |
| **Uncertain labels.** 12 of the 19 training Silage labels rest on a planting record only. | The chosen setup is retrained without them (and without the 10 misfitting outlines); the test set does not change. |

**What this can show:** whether Prithvi matches the baseline on a fair split, which field-years it gets wrong,
whether finer frames help, and whether the answer depends on the least certain labels.

**What it cannot show:** that Prithvi is better than the baseline. The test has 44 field-years (12 Silage); section 8
computes that even a perfect score would not be a significant improvement. It also says nothing about farms other
than BARC.

## 1. Hypothesis and data

**Hypothesis** (stated before modelling, same as `research/grain_vs_silage/README.md`): at BARC silage corn is cut
green in late August to early September, leaving a sudden fall in greenness and a bare field; grain corn dries down
slowly and leaves residue. A model that sees the field every two weeks from August to November should separate the two.
An early fall in NDVI is evidence about harvest timing, not a label.

| Source | Use | Access |
|---|---|---|
| Sentinel-2 L2A, Landsat 8/9 Collection 2 Level-2 | Six-band 30 m image stacks, June-November 2017-2023 | Microsoft Planetary Computer, read by `prithvi_chips.py`; queries and failures in `<chips>/<year>.provenance.json` |
| Prithvi-EO-2.0 (IBM and NASA, Apache 2.0) | Pretrained encoder | Hugging Face `ibm-nasa-geospatial/Prithvi-EO-2.0-*-TL` |
| BARC evaluation file | Test labels: 44 field-years, 2021-2023 | `04_Evaluation/BARC_Corn.2021-2023.gpkg` |
| BARC farm records (Kraatz 2023, Zenodo 8087301, CC BY 4.0) | Training labels: 83 field-years, 2017-2020 | `classifier/labels/` (sources, checksums, checks) |

Blank labels in the evaluation file are unlabeled, not negatives; only labeled field-years are used.

## 2. Settings and checks

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path

HERE = Path.cwd()
ROOT = next(p for p in [HERE, *HERE.parents] if (p / "classifier" / "prithvi" / "prithvi_train.py").exists())
PRITHVI = ROOT / "classifier" / "prithvi"
env = lambda name, default: os.environ.get(name, default)

CHIPS = Path(env("PRITHVI_CHIPS", "/home/keeboh/work/prithvi/chips"))   # <year>.npz image stacks from prithvi_chips.py
OUT = Path(env("PRITHVI_OUT", str(PRITHVI / "results" / "year_split")))
if Path("/home/keeboh/hf").exists():
    os.environ.setdefault("HF_HOME", "/home/keeboh/hf")                  # cached Prithvi checkpoints
BUILD_MISSING_CHIPS = False     # True: run prithvi_chips.py for any missing year (5.5-7.5 min per year, needs network)

# Candidates. Decided before any result was seen; change them here, not after looking at the test.
SIZES = env("PRITHVI_SIZES", "100M,300M").split(",")      # frozen encoders
SCALE = 2                                                 # 30 m chip upsampled to 15 m: one 16 x 16 patch is 240 m, about one field
FRAME_SETS = [4, 8]                                       # 4 = the earlier windows, 8 = two-week windows
KS = [8, 16]                                              # PCA components of the frozen embedding
C_HEAD = 0.5                                              # logistic regression C, as in the baseline
RUN_FINETUNE = env("PRITHVI_FINETUNE", "1") == "1"
FT_SIZE = env("PRITHVI_FT_SIZE", "100M")                  # fine-tuned encoder (last two blocks)
EPOCHS = int(env("PRITHVI_EPOCHS", "40"))                 # upper limit; the number used is chosen on 2017-2020
SEEDS = [int(s) for s in env("PRITHVI_SEEDS", "0,1,2,3,4").split(",")]
P_DROP = 0.25                                             # date dropout: chance that one scene-day is left out
N_AUG = int(env("PRITHVI_N_AUG", "8"))                    # dropout versions per training field-year
N_STAB = int(env("PRITHVI_N_STAB", "20"))                 # dropout versions per test field-year (stability check)

# First pass on a new machine: set QUICK = True and Run All. It runs every cell with the smallest encoder and a few
# epochs to check that the code and inputs work, and writes to its own folder. Its numbers are NOT results.
QUICK = env("PRITHVI_QUICK", "0") == "1"
if QUICK:
    SIZES, FT_SIZE, EPOCHS, SEEDS, N_AUG, N_STAB = ["tiny"], "tiny", 3, [0, 1], 2, 3
    OUT = OUT.parent / (OUT.name + "_quick")
    print("QUICK mode: a code check only. Set QUICK = False for the real run.")

need = {"timm": "timm", "einops": "einops", "huggingface_hub": "huggingface_hub", "geopandas": "geopandas",
        "rasterio": "rasterio", "sklearn": "scikit-learn", "scipy": "scipy", "shapely": "shapely",
        "matplotlib": "matplotlib"}
if importlib.util.find_spec("torch") is None:
    raise SystemExit("PyTorch is not installed in this kernel. Use the GPU environment (see classifier/README.md).")
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("installing", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
import copy, json, platform, time, warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from rasterio.transform import Affine
from scipy.stats import binomtest
from shapely import make_valid
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(PRITHVI))
import prithvi_train as pt     # field_inputs, load_encoder, FieldClassifier, embed_all, predict, dihedral, make_logreg, ...
import prithvi_embed as pe     # FRAMES (the two-week windows), fill_gaps

T0 = time.time()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 200)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (no GPU found: this will be slow)")

fy = pd.read_csv(PRITHVI / "field_years.csv")
lack = {"label_basis", "outline_fit", *pt.SMALL} - set(fy.columns)
if lack:
    raise SystemExit(f"field_years.csv lacks {sorted(lack)}: upload the current classifier/prithvi/field_years.csv")
years = sorted(fy.year.unique().tolist())
absent = [yr for yr in years if not (CHIPS / f"{yr}.npz").exists()]
if absent and BUILD_MISSING_CHIPS:
    CHIPS.mkdir(parents=True, exist_ok=True)
    for yr in absent:
        subprocess.check_call([sys.executable, str(PRITHVI / "prithvi_chips.py"), "--out", str(CHIPS),
                               "--years", str(yr), "--workers", "6"])
    absent = [yr for yr in years if not (CHIPS / f"{yr}.npz").exists()]
if absent:
    raise SystemExit(f"no image stack for {absent} in {CHIPS}. Set CHIPS to the folder written by prithvi_chips.py, "
                     "or set BUILD_MISSING_CHIPS = True to build them.")
print(f"{len(fy)} labeled field-years, image stacks for {years} in {CHIPS}")

## 3. Labeled field-years by year and class

Training years (2017-2020) come from the farm records; test years (2021-2023) are the official labels.
`label_basis` says what a farm-record label rests on: a harvest record, or only a planting record (the plan).

In [ ]:
fy["y"] = (fy.label == "Silage").astype(int)
fy["farmkey"] = fy.farm.str.replace(" ", "", regex=False)
N, idx, y, yr = len(fy), np.arange(len(fy)), fy.y.to_numpy(), fy.year.to_numpy()
H = fy[pt.SMALL].to_numpy(float)                      # the 5 hand features of the baseline

display(pd.crosstab(fy.year, fy.label, margins=True))
display(pd.crosstab([fy.label_source, fy.label_basis], fy.label, margins=True))
print("distinct fields:", fy.field_id.nunique(), "| field-years by farm:", fy.farmkey.value_counts().to_dict())

## 4. Splits, and the baseline must reproduce

Every split holds out whole years; none is a random field split. They are the six splits of
`research/grain_vs_silage/model_comparison.py`, rebuilt here from `field_years.csv`.

| Split | Role |
|---|---|
| Hold out one year, 2017-20 only | **Choosing the candidate.** Never touches a 2021-2023 label. |
| Train 2017-20, test 2021-23 | **Headline test** (44 official labels, out of time). |
| Train 2017-20 other farms, test Central Farm 2021-23 | Partial spatial holdout (11 of the 12 test Silage are on Central Farm). |
| Train 2017-20 unseen fields only, test 2021-23 | No test field is in training. A chip shows the field's surroundings, so a model could recognise a field instead of its harvest. |
| Hold out one year, 2017-23; hold out one year, 2021-23 | More test field-years, but they use official labels for training. |

Before anything new is trusted, the 5-feature logistic regression has to give its published numbers of correct
field-years on all six splits.

In [ ]:
old, new = yr <= 2020, yr >= 2021
central = (fy.farmkey == "CentralFarm").to_numpy()
seen = fy.field_id.isin(fy.field_id[new]).to_numpy()            # fields that also appear in 2021-2023
CHOICE, MAIN = "Hold out one year, 2017-20 only (choice)", "Train 2017-20, test 2021-23"
SPLITS = {
    CHOICE: [(idx[old & (yr != k)], idx[old & (yr == k)]) for k in (2017, 2018, 2019, 2020)],
    MAIN: [(idx[old], idx[new])],
    "Train 2017-20 other farms, test Central Farm 2021-23": [(idx[old & ~central], idx[new & central])],
    "Train 2017-20 unseen fields only, test 2021-23": [(idx[old & ~seen], idx[new])],
    "Hold out one year, 2017-23": [(idx[yr != k], idx[yr == k]) for k in range(2017, 2024)],
    "Hold out one year, 2021-23": [(idx[new & (yr != k)], idx[new & (yr == k)]) for k in (2021, 2022, 2023)],
}
PUBLISHED = dict(zip(SPLITS, (68, 39, 24, 40, 113, 42)))        # correct field-years, hand 5 / logistic


def predict_split(c, folds):
    """Held-out rows, their Silage probability (mean over the candidate's seeds) and the per-seed probabilities."""
    rows, per_seed = [], []
    for tr, te in folds:
        per_seed.append(np.stack([c["fit"](tr, s)(te) for s in c["seeds"]]))
        rows.append(te)
    per_seed = np.concatenate(per_seed, axis=1)
    return np.concatenate(rows), per_seed.mean(0), per_seed


def baseline_fit(tr, seed=0):
    m = pt.make_logreg().fit(H[tr], y[tr])
    return lambda rows, Xn=None: m.predict_proba(H[rows])[:, 1]


BASE = dict(name="hand 5 / logistic (baseline)", kind="baseline", fit=baseline_fit, seeds=[0], inputs=5)
base_pred, gate = {}, []
for name, folds in SPLITS.items():
    te, p, _ = predict_split(BASE, folds)
    base_pred[name] = (te, p)
    got = int(((p >= 0.5) == (y[te] == 1)).sum())
    gate.append({"split": name, "train field-years": "/".join(str(len(tr)) for tr, _ in folds), "test": len(te),
                 "Silage in test": int(y[te].sum()), "correct": got, "published": PUBLISHED[name]})
gate = pd.DataFrame(gate)
display(gate)
if (gate.correct != gate.published).any():
    raise SystemExit("the baseline does not reproduce its published numbers: stop and check field_years.csv")
print("baseline reproduces on all six splits")

## 5. Inputs: frames from the image stacks

For each field-year: a 48 x 48 pixel chip (1.44 km at 30 m) centred on the field, and one frame per time window, each
the per-pixel median of the cloud-free readings in that window. A pixel with no clear reading in a window takes the
value of the nearest window that has one (`prithvi_embed.fill_gaps`).

Two differences from `prithvi_train.py`, both needed for date dropout (section 7):

- The daily crops are kept, so the frames can be rebuilt with some scene-days left out. The cell checks that the
  frames built here equal those of `pt.field_inputs` exactly.
- Every frame is given its window's year and mid-date. `pt.field_inputs` leaves a window without a clear reading at
  year 0, day 0, which is wrong once the frame has been gap-filled.

The table and figures are input checks: how many clear scene-days each window has (date dropout can empty a
window with few), and whether the frames show the expected signal (Silage NDVI falling earlier than Grain).

In [ ]:
SILAGE, GRAIN, INK, MUTED, SURFACE = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c9c8c2",
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
                     "axes.grid": True, "grid.color": "#e7e6e1", "grid.linewidth": 0.8, "axes.axisbelow": True,
                     "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "legend.frameon": False, "figure.dpi": 110})

geoms = gpd.GeoSeries.from_wkt(fy.wkt, crs=pt.CRS).apply(make_valid)
half = pt.CHIP // 2
crops, doys, stack_rows = [None] * N, [None] * N, []
for year in years:
    z = np.load(CHIPS / f"{year}.npz")
    x, tr = z["x"], Affine(*z["transform"])
    doy = pd.to_datetime(z["dates"]).day_of_year.values
    for i in np.flatnonzero(yr == year):                         # same crop as pt.field_inputs
        c = geoms.iloc[i].centroid
        col, row = int((c.x - tr.c) // pt.RES), int((tr.f - c.y) // pt.RES)
        crops[i], doys[i] = x[:, :, row - half:row + half, col - half:col + half].copy(), doy
    info = {"year": year, "scene-days": len(doy)}
    if "sensors" in z.files:
        info.update(pd.Series(z["sensors"]).value_counts().to_dict())
    prov = CHIPS / f"{year}.provenance.json"
    if prov.exists():
        py = json.loads(prov.read_text())["years"][str(year)]
        info.update({"read": py["scene_days_read"], "kept": py["scene_days_kept"], "failed": len(py["failures"])})
    stack_rows.append(info)
print("Image stacks (scene-days kept; 'failed' = reads that failed and are missing):")
display(pd.DataFrame(stack_rows).set_index("year"))


def window_doys(year, windows):
    return [(pd.Timestamp(f"{year}-{a}").day_of_year, pd.Timestamp(f"{year}-{b}").day_of_year) for a, b in windows]


def frames(i, windows, keep=None):
    """(6, T, 48, 48) frames of field-year i: per-pixel median of the clear readings in each window (NaN if none).
    keep: optional boolean per scene-day; days marked False are left out (date dropout)."""
    out = np.full((6, len(windows), pt.CHIP, pt.CHIP), np.nan, np.float32)
    for t, (a, b) in enumerate(window_doys(yr[i], windows)):
        sel = (doys[i] >= a) & (doys[i] <= b)
        if keep is not None:
            sel &= keep
        if sel.any():
            c = crops[i][sel].astype(np.float32)
            c[c == pt.NODATA] = np.nan
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                out[:, t] = np.nanmedian(c, axis=0)
    return out


INP = {}
for n in FRAME_SETS:
    windows = pe.FRAMES[n]
    pt.WINDOWS = windows                                         # pt.field_inputs reads its windows from here
    X_pt, M, _, LC = pt.field_inputs(fy, CHIPS)
    X = np.stack([frames(i, windows) for i in range(N)])
    assert np.allclose(X, X_pt, equal_nan=True), "frames differ from pt.field_inputs"
    TC = np.array([[(yr[i], (a + b) / 2) for a, b in window_doys(yr[i], windows)] for i in range(N)], np.float32)
    INP[n] = dict(raw=X, X=pe.fill_gaps(X), M=M, TC=TC, LC=LC, windows=windows)
pt.WINDOWS = pe.FRAMES[4]
print("frames equal pt.field_inputs for", FRAME_SETS, "frame sets")

In [ ]:
# Clear scene-days per window: a day counts for a field when at least 60% of its pixels are cloud-free.
n8 = max(FRAME_SETS)
inside = INP[n8]["M"] > 0.5
inside[inside.reshape(N, -1).sum(1) == 0] = (INP[n8]["M"] > 0)[inside.reshape(N, -1).sum(1) == 0]
clear = [(crops[i][:, 0][:, inside[i]] != pt.NODATA).mean(1) >= 0.6 for i in range(N)]
wl = [f"{int(a[:2])}/{a[3:]}-{int(b[:2])}/{b[3:]}" for a, b in INP[n8]["windows"]]
cnt = np.array([[int((clear[i] & (doys[i] >= a) & (doys[i] <= b)).sum())
                 for a, b in window_doys(yr[i], INP[n8]["windows"])] for i in range(N)])
med = pd.DataFrame(cnt, columns=wl).groupby(yr).median()

fig, ax = plt.subplots(figsize=(1.15 * len(wl) + 1.5, 0.42 * len(med) + 1.3))
ax.imshow(med.values, cmap="Blues", vmin=0, vmax=max(6, med.values.max()), aspect="auto")
for (r, c), v in np.ndenumerate(med.values):
    ax.text(c, r, f"{v:g}", ha="center", va="center", color="white" if v >= 4 else INK, fontsize=9)
ax.set_xticks(range(len(wl)), wl, fontsize=8); ax.set_yticks(range(len(med)), med.index)
ax.grid(False); ax.set_title(f"Median clear scene-days per field-year, by year and {n8}-frame window")
plt.tight_layout(); plt.show()

empty = pd.DataFrame({"no clear day (share of field-years)": (cnt == 0).mean(0),
                      f"emptied by date dropout, p={P_DROP} (expected share)": (P_DROP ** cnt).mean(0)}, index=wl).round(3)
display(empty.T)

In [ ]:
# Median field NDVI per frame and class (all labeled field-years): does the frame set resolve the silage cut?
fig, axes = plt.subplots(1, len(FRAME_SETS), figsize=(5.6 * len(FRAME_SETS), 3.6), sharey=True, squeeze=False)
for ax, n in zip(axes[0], FRAME_SETS):
    X, M = INP[n]["raw"], INP[n]["M"]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        ndvi = (X[:, 3] - X[:, 2]) / (X[:, 3] + X[:, 2])
        field = np.nanmean(np.where((M > 0.5)[:, None], ndvi, np.nan), axis=(2, 3))            # (N, T)
        mid = [sum(d) / 2 for d in window_doys(2021, INP[n]["windows"])]
        for lab, colour in (("Grain", GRAIN), ("Silage", SILAGE)):
            v = np.nanmedian(field[fy.label.values == lab], axis=0)
            ax.plot(mid, v, color=colour, lw=2, marker="o", ms=6, label=lab)
            ax.annotate(lab, (mid[-1], v[-1]), xytext=(6, 0), textcoords="offset points", va="center", color=INK)
    ax.set_xticks(mid, [pd.Timestamp("2021-01-01") + pd.Timedelta(days=d - 1) for d in mid])
    ax.set_xticklabels([(pd.Timestamp("2021-01-01") + pd.Timedelta(days=d - 1)).strftime("%b %d") for d in mid],
                       rotation=45, ha="right", fontsize=8)
    ax.set_xlim(mid[0] - 6, mid[-1] + 22); ax.set_title(f"{n} frames"); ax.set_xlabel("window mid-date")
axes[0][0].set_ylabel("median field NDVI"); axes[0][0].legend(loc="lower left")
plt.tight_layout(); plt.show()

## 6. Metrics

Silage is the positive class. With TP, FN, FP, TN counted at a probability cut (0.5 unless stated) over $N$ test
field-years:

| Metric | Definition | Reads as |
|---|---|---|
| Predicted positive rate | $(TP+FP)/N$ | share of field-years the model calls Silage |
| Actual positive rate | $(TP+FN)/N$ | share that are Silage; if the predicted rate is higher, the model over-calls Silage |
| Silage recall (true positive rate) | $TP/(TP+FN)$ | Silage fields caught |
| False positive rate | $FP/(FP+TN)$ | Grain fields called Silage |
| Silage precision | $TP/(TP+FP)$ | calls of Silage that are right |
| Grain recall, Grain precision | $TN/(TN+FP)$, $TN/(TN+FN)$ | the same for Grain |
| Balanced accuracy | (Silage recall + Grain recall) / 2 | accuracy that does not reward calling everything Grain |
| ROC-AUC, PR-AUC | ranking quality, independent of the cut | AUC can be high while the cut is off |
| Log loss | $-\frac{1}{N}\sum_i [y_i\log p_i+(1-y_i)\log(1-p_i)]$ | how good the probabilities are |

Uncertainty: a 95% Wilson interval on each proportion, $\frac{\hat p + z^2/2n \pm z\sqrt{\hat p(1-\hat p)/n + z^2/4n^2}}{1+z^2/n}$
with $z=1.96$. Two models on the same test rows are compared with the exact McNemar test: a two-sided binomial test
on the field-years where exactly one of them is right.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return "n/a"
    p, den = k / n, 1 + z * z / n
    c, h = (p + z * z / (2 * n)) / den, z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return f"{max(0.0, c - h):.2f}-{min(1.0, c + h):.2f}"


def metrics(yv, p, cut=0.5):
    pred = p >= cut
    tp, fn = int((pred & (yv == 1)).sum()), int((~pred & (yv == 1)).sum())
    fp, tn = int((pred & (yv == 0)).sum()), int((~pred & (yv == 0)).sum())
    n, div = len(yv), lambda a, b: round(a / b, 3) if b else np.nan
    both = 0 < yv.sum() < n
    return {"n": n, "Silage": tp + fn, "Grain": fp + tn, "TP": tp, "FN": fn, "FP": fp, "TN": tn,
            "correct": tp + tn, "accuracy": div(tp + tn, n), "accuracy 95%": wilson(tp + tn, n),
            "balanced accuracy": round((tp / (tp + fn) + tn / (tn + fp)) / 2, 3) if both else np.nan,
            "predicted positive rate": div(tp + fp, n), "actual positive rate": div(tp + fn, n),
            "Silage recall (TPR)": div(tp, tp + fn), "Silage recall 95%": wilson(tp, tp + fn),
            "Silage precision": div(tp, tp + fp), "false positive rate": div(fp, fp + tn),
            "Grain recall": div(tn, tn + fp), "Grain recall 95%": wilson(tn, tn + fp),
            "Grain precision": div(tn, tn + fn),
            "ROC-AUC": round(roc_auc_score(yv, p), 3) if both else np.nan,
            "PR-AUC": round(average_precision_score(yv, p), 3) if both else np.nan,
            "log loss": round(log_loss(yv, np.clip(p, 1e-6, 1 - 1e-6), labels=[0, 1]), 3)}


def mcnemar(yv, pa, pb, cut_a=0.5, cut_b=0.5):
    """a = baseline, b = the other model. Returns (only b right, only a right, exact two-sided p)."""
    ca, cb = (pa >= cut_a) == (yv == 1), (pb >= cut_b) == (yv == 1)
    only_b, only_a = int((cb & ~ca).sum()), int((ca & ~cb).sum())
    return only_b, only_a, round(binomtest(only_b, only_a + only_b).pvalue, 3) if only_a + only_b else 1.0


def best_cut(yv, p):
    """Cut with the highest balanced accuracy on out-of-fold training predictions; ties go to the one nearest 0.5."""
    s = np.unique(p)
    cuts = (s[:-1] + s[1:]) / 2 if len(s) > 1 else np.array([0.5])
    ba = np.array([((p[yv == 1] >= c).mean() + (p[yv == 0] < c).mean()) / 2 for c in cuts])
    best = cuts[np.isclose(ba, ba.max())]
    return float(best[np.argmin(np.abs(best - 0.5))])


def confusion(yv, p, cut=0.5):
    pred = p >= cut
    return pd.DataFrame([[int((~pred & (yv == 0)).sum()), int((pred & (yv == 0)).sum())],
                         [int((~pred & (yv == 1)).sum()), int((pred & (yv == 1)).sum())]],
                        index=["true Grain", "true Silage"], columns=["predicted Grain", "predicted Silage"])


def by_year(te, p, cut=0.5):
    d = pd.DataFrame({"year": yr[te], "y": y[te], "pred": (p >= cut).astype(int)})
    return d.groupby("year").agg(n=("y", "size"), Silage=("y", "sum"), predicted_Silage=("pred", "sum"),
                                 correct=("y", lambda s: int((s == d.loc[s.index, "pred"]).sum())))

## 7. Candidates

Training has 83 field-years, 19 of them Silage, so the inputs to anything that is fitted are kept few and the list of
candidates short. All use the 15 m scale (one patch is about one field).

- **Frozen encoder (16 candidates).** Encoder size (100M, 300M) x frames (4, 8). The embedding (frames x encoder
  width, 3,000 to 8,000 numbers) is reduced inside each training fold to 8 or 16 principal components, alone or next
  to the 5 hand features, then a logistic regression with the baseline's settings. Same head as the team's
  `evaluate.py`.
- **Fine-tuned (4 candidates, an experiment).** 100M encoder, last two blocks and a linear head trained
  (settings of `pt.finetune`), frames (4, 8), with and without date dropout. Millions of trained weights on 83
  field-years: it rests entirely on the pretrained weights and a low learning rate. There is no early stopping on
  held-out data here; the number of epochs is a setting chosen on 2017-2020 like any other.
- **Date dropout** (fine-tuning only). Which days are cloud-free differs by field and year. Each scene-day is left
  out with probability 0.25 and the frames are rebuilt, giving 8 versions of every training field-year; each epoch
  uses one at random. This mimics a nuisance the model should ignore. It does not add labels: there are still 19
  Silage field-years, and it is applied to training rows only.

In [ ]:
ENC, NORM = {}, {}


def encoder(size, n):
    """Pretrained encoder for `size` and `n` frames. Never trained in place: fine-tuning works on a copy."""
    if (size, n) not in ENC:
        enc, cfg, info = pt.load_encoder(size, n, pt.CHIP * SCALE, DEVICE)
        NORM.setdefault("cfg", cfg)
        assert cfg["mean"] == NORM["cfg"]["mean"] and cfg["std"] == NORM["cfg"]["std"]
        ENC[(size, n)] = (enc, info)
    return ENC[(size, n)]


for size in dict.fromkeys(SIZES + ([FT_SIZE] if RUN_FINETUNE else [])):
    for n in FRAME_SETS:
        encoder(size, n)
normalise = lambda X: pt.normalise(X, NORM["cfg"])
DATA = {n: (normalise(INP[n]["X"]), INP[n]["M"], INP[n]["TC"], INP[n]["LC"]) for n in FRAME_SETS}


def embed(size, n, Xn, rows):
    """Frozen embedding of normalised frames Xn for field-years `rows` (mean over the 8 rotations and flips)."""
    model = pt.FieldClassifier(encoder(size, n)[0], n, SCALE).to(DEVICE)
    _, M, TC, LC = DATA[n]
    return pt.embed_all(model, (Xn, M[rows], TC[rows], LC[rows]), np.arange(len(rows)), DEVICE)


def dropped(rows, n, rng):
    """Normalised, gap-filled frames of `rows` with each scene-day left out with probability P_DROP."""
    X = np.stack([frames(i, INP[n]["windows"], rng.random(len(doys[i])) > P_DROP) for i in rows])
    return normalise(pe.fill_gaps(X))


with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    Z = {(size, n): embed(size, n, DATA[n][0], idx) for size in SIZES for n in FRAME_SETS}
rng_aug = np.random.default_rng(0)
XA = ({n: np.stack([dropped(idx, n, rng_aug) for _ in range(N_AUG)]).astype(np.float16) for n in FRAME_SETS}
      if RUN_FINETUNE else {})
print({f"{s} f{n}": z.shape for (s, n), z in Z.items()}, f"| {time.time() - T0:.0f} s")

In [ ]:
def head(n_emb, n_hand, k):
    """Impute, standardise, PCA(k) on the embedding columns only, then logistic regression (as in evaluate.py)."""
    parts = [("emb", make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), PCA(k, svd_solver="full")),
              list(range(n_emb)))]
    if n_hand:
        parts.append(("hand", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
                      list(range(n_emb, n_emb + n_hand))))
    return make_pipeline(ColumnTransformer(parts),
                         LogisticRegression(C=C_HEAD, class_weight="balanced", max_iter=5000))


def frozen_candidate(size, n, k, hand):
    Zc = Z[(size, n)]
    feats = lambda Zr, rows: np.hstack([Zr, H[rows]]) if hand else Zr

    def fit(tr, seed=0):
        clf = head(Zc.shape[1], 5 * hand, k).fit(feats(Zc[tr], tr), y[tr])
        return lambda rows, Xn=None: clf.predict_proba(
            feats(Zc[rows] if Xn is None else embed(size, n, Xn, rows), rows))[:, 1]

    return dict(name=f"{size} f{n} frozen, {'hand 5 + ' if hand else ''}PCA {k}", kind="frozen", size=size, frames=n,
                fit=fit, seeds=[0], inputs=k + 5 * hand, embedding_width=Zc.shape[1])


def ft_fit(c, tr, seed, epochs, trace_rows=None, bs=16, lr_enc=2e-5, lr_head=1e-3):
    """Train the last two encoder blocks and a linear head on rows `tr` for `epochs` epochs (settings of pt.finetune,
    without early stopping). Returns a predictor and, if trace_rows is given, their probabilities after every epoch."""
    n = c["frames"]
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    enc = copy.deepcopy(encoder(c["size"], n)[0])
    model = pt.FieldClassifier(enc, n, SCALE).to(DEVICE)
    for p in enc.parameters():
        p.requires_grad = False
    train_enc = list(enc.blocks[-2:]) + [enc.norm]
    for mod in train_enc:
        for p in mod.parameters():
            p.requires_grad = True
    c["trained_weights"] = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt = torch.optim.AdamW([{"params": [p for mod in train_enc for p in mod.parameters()], "lr": lr_enc},
                             {"params": list(model.head.parameters()), "lr": lr_head}], weight_decay=0.05)
    pos_weight = torch.tensor(float((y[tr] == 0).sum() / max(1, (y[tr] == 1).sum())), device=DEVICE)
    X0, M, TC, LC = DATA[n]
    predictor = lambda rows, Xn=None: pt.predict(
        model, (X0[rows] if Xn is None else Xn, M[rows], TC[rows], LC[rows]), np.arange(len(rows)), DEVICE)
    trace = []
    for ep in range(epochs):
        model.train()
        enc.eval()                                               # no dropout or drop-path in the encoder
        perm = rng.permutation(tr)
        for s in range(0, len(perm), bs):
            j = perm[s:s + bs]
            xb = XA[n][rng.integers(N_AUG, size=len(j)), j].astype(np.float32) if c["aug"] else X0[j]
            x, m = pt.dihedral(torch.from_numpy(xb).to(DEVICE), torch.from_numpy(M[j]).to(DEVICE), int(rng.integers(8)))
            tc, lc = torch.from_numpy(TC[j]).to(DEVICE), torch.from_numpy(LC[j]).to(DEVICE)
            with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
                logit = model(x, m, tc, lc).float()
            loss = F.binary_cross_entropy_with_logits(logit, torch.from_numpy(y[j]).float().to(DEVICE),
                                                      pos_weight=pos_weight)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        if trace_rows is not None:
            trace.append(predictor(trace_rows))
    model.eval()
    return predictor, np.array(trace)


def finetune_candidate(size, n, aug):
    c = dict(name=f"{size} f{n} fine-tuned{', date dropout' if aug else ''}", kind="finetune", size=size, frames=n,
             aug=aug, seeds=SEEDS, epochs=None, embedding_width=n * encoder(size, n)[1]["embed_dim"])
    c["fit"] = lambda tr, seed: ft_fit(c, tr, seed, c["epochs"])[0]
    return c


CANDS = [frozen_candidate(size, n, k, hand) for size in SIZES for n in FRAME_SETS for hand in (False, True) for k in KS]
if RUN_FINETUNE:
    CANDS += [finetune_candidate(FT_SIZE, n, aug) for n in FRAME_SETS for aug in (False, True)]
print(len(CANDS), "candidates")

## 8. Choosing a candidate on 2017-2020 only

Each candidate is scored on the four held-out years of 2017-2020 (83 field-years, 19 Silage), pooled. The one with the
lowest log loss is chosen (ties: higher AUC). For a fine-tuned candidate the number of epochs is whichever gives the
lowest pooled log loss on the same split. No 2021-2023 label is read in this section.

The last column of the first table is the number of Silage training field-years per fitted input; 10 or more is the
usual rule of thumb, and every candidate is below it. The baseline is listed for reference and cannot be chosen.

In [ ]:
warnings.filterwarnings("ignore", message=".*CUDA is not available.*")
folds = SPLITS[CHOICE]
te_c = np.concatenate([te for _, te in folds])
rows = []
for c in [BASE] + CANDS:
    if c["kind"] == "finetune":
        P = np.concatenate([ft_fit(c, tr, SEEDS[0], EPOCHS, trace_rows=te)[1] for tr, te in folds], axis=1)
        ll = [log_loss(y[te_c], np.clip(p, 1e-6, 1 - 1e-6), labels=[0, 1]) for p in P]
        c["epochs"], p = int(np.argmin(ll)) + 1, P[int(np.argmin(ll))]
        c["epoch_curve"] = np.round(ll, 3).tolist()
    else:
        _, p, _ = predict_split(c, folds)
    c["choice_p"] = p
    m = metrics(y[te_c], p)
    rows.append({"candidate": c["name"], "fitted inputs or weights": c.get("inputs") or c["trained_weights"],
                 "Silage per input": round(int(y[old].sum()) / c["inputs"], 2) if c.get("inputs") else "<0.001",
                 "epochs": c.get("epochs"), "log loss": m["log loss"], "ROC-AUC": m["ROC-AUC"],
                 "correct": m["correct"], "Silage caught": f"{m['TP']}/{m['Silage']}",
                 "Grain called Silage": f"{m['FP']}/{m['Grain']}"})
    print(f"{c['name']}: log loss {m['log loss']}, {time.time() - T0:.0f} s", flush=True)
choice = pd.DataFrame(rows)
choice.insert(0, "rank", choice["log loss"].rank(method="first").astype(int))
choice = choice.sort_values(["log loss", "ROC-AUC"], ascending=[True, False]).reset_index(drop=True)
display(choice)

CHOSEN = next(c for name in choice.candidate for c in CANDS if c["name"] == name)
print(f"\nChosen on 2017-2020: {CHOSEN['name']}" + (f" ({CHOSEN['epochs']} epochs)" if CHOSEN["kind"] == "finetune" else ""))
top = choice[choice.candidate != BASE["name"]].head(5)
print(f"Log loss of the five best candidates: {top['log loss'].min()} to {top['log loss'].max()}; "
      f"correct: {top.correct.min()} to {top.correct.max()} of {len(te_c)}. One field-year is "
      f"{100 / len(te_c):.1f} points, so a small gap between them is not evidence that one is better.")

## 9. Test: scored once, for the chosen candidate and the baseline

Both are fitted on the same training rows and scored on the same held-out rows of each split. A fine-tuned
candidate is fitted once per seed and its probabilities are averaged; the per-seed counts are printed.

Two cuts are shown: 0.5, and each model's own cut chosen on its 2017-2020 out-of-fold probabilities (highest balanced
accuracy). The second one never sees a 2021-2023 label either.

In [ ]:
TEST_SPLITS = [s for s in SPLITS if s != CHOICE]
cut = {"baseline": best_cut(y[te_c], BASE["choice_p"]), "Prithvi": best_cut(y[te_c], CHOSEN["choice_p"])}
pred_rows, met_rows, mc_rows, test_pred = [], [], [], {}
for split in TEST_SPLITS:
    te, pb = base_pred[split]
    te2, pp, per_seed = predict_split(CHOSEN, SPLITS[split])
    assert (te == te2).all()
    test_pred[split] = (te, pb, pp, per_seed)
    pred_rows.append(fy.loc[te, ["field_id", "farm", "field", "year", "label", "label_basis"]]
                     .assign(split=split, p_baseline=pb.round(4), p_prithvi=pp.round(4)))
    for model, p in (("baseline", pb), ("Prithvi", pp)):
        met_rows.append({"split": split, "model": model, "cut": 0.5, **metrics(y[te], p)})
        if split == MAIN:
            met_rows.append({"split": split, "model": model, "cut": round(cut[model], 3), **metrics(y[te], p, cut[model])})
    ob, oa, pv = mcnemar(y[te], pb, pp)
    mc_rows.append({"split": split, "baseline correct": int(((pb >= 0.5) == (y[te] == 1)).sum()),
                    "Prithvi correct": int(((pp >= 0.5) == (y[te] == 1)).sum()), "of": len(te),
                    "only Prithvi right": ob, "only baseline right": oa, "McNemar p": pv})
    print(f"{split}: done, {time.time() - T0:.0f} s", flush=True)
PRED, MET, MC = pd.concat(pred_rows, ignore_index=True), pd.DataFrame(met_rows), pd.DataFrame(mc_rows)

te, pb, pp, per_seed = test_pred[MAIN]
print(f"\n=== {MAIN}: {len(te)} field-years ({int((y[te] == 0).sum())} Grain, {int(y[te].sum())} Silage); "
      f"Prithvi = {CHOSEN['name']} ===")
show = ["model", "cut", "correct", "accuracy", "accuracy 95%", "balanced accuracy", "predicted positive rate",
        "actual positive rate", "Silage recall (TPR)", "Silage recall 95%", "Silage precision", "false positive rate",
        "Grain recall", "Grain recall 95%", "Grain precision", "ROC-AUC", "PR-AUC", "log loss"]
display(MET[MET.split == MAIN][show].set_index(["model", "cut"]).T)
for model, p in (("baseline", pb), ("Prithvi", pp)):
    print(f"\nConfusion matrix, {model}, cut 0.5:"); display(confusion(y[te], p))
print("\nBy test year, cut 0.5 (predicted_Silage against Silage shows over- or under-calling in that year):")
display(pd.concat({"baseline": by_year(te, pb), "Prithvi": by_year(te, pp)}, axis=1))
if len(per_seed) > 1:
    print("Prithvi correct per fine-tuning seed:", [int(((p >= 0.5) == (y[te] == 1)).sum()) for p in per_seed],
          "| mean probability over seeds:", int(((pp >= 0.5) == (y[te] == 1)).sum()))

In [ ]:
print("Every year-based split, cut 0.5:")
display(MET[MET.cut == 0.5][["split", "model", "n", "Silage", "correct", "predicted positive rate",
                             "actual positive rate", "Silage recall (TPR)", "false positive rate", "Silage precision",
                             "Grain precision", "ROC-AUC", "log loss"]].set_index(["split", "model"]))
print("Prithvi against the baseline (exact McNemar on the field-years where they disagree):")
display(MC.set_index("split"))

# What this test can and cannot show.
errors = int(((pb >= 0.5) != (y[te] == 1)).sum())
best_p = binomtest(errors, errors).pvalue if errors else 1.0
need = next(k for k in range(1, 60) if binomtest(k, k).pvalue < 0.05)
print(f"Power of the headline test: the baseline is wrong on {errors} of {len(te)}. A model that got all {len(te)} "
      f"right would win {errors}-0, McNemar p = {best_p:.4f}. A significant win at 0.05 needs {need}-0, so this test "
      f"{'cannot' if errors < need else 'can only just'} show that any model beats the baseline. Read the tables "
      "above as 'about the same or not', and look at which field-years differ.")

In [ ]:
# Silage probability of every test field-year, by year: the baseline on the left, Prithvi on the right.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
ty = sorted(set(yr[te]))
jit = np.random.default_rng(0).uniform(-0.28, 0.28, len(te))
for ax, (title, p) in zip(axes, ((BASE["name"], pb), (CHOSEN["name"], pp))):
    for lab, colour, mark in ((0, GRAIN, "o"), (1, SILAGE, "D")):
        s = y[te] == lab
        ax.scatter(p[s], [ty.index(v) for v in yr[te][s]] + jit[s], s=46, color=colour, marker=mark,
                   edgecolor=SURFACE, linewidth=1.2, label=("Grain", "Silage")[lab], zorder=3)
    ax.axvline(0.5, color=MUTED, lw=1, ls="--", zorder=2)
    ax.set_yticks(range(len(ty)), ty); ax.set_xlim(-0.03, 1.03); ax.set_xlabel("predicted probability of Silage")
    ax.set_title(f"{title}: {int(((p >= 0.5) == (y[te] == 1)).sum())} of {len(te)} correct")
    ax.grid(axis="y", visible=False)
axes[0].legend(loc="center", ncols=2, bbox_to_anchor=(0.5, -0.32)); axes[0].set_ylabel("test year")
fig.suptitle("Train 2017-2020, test 2021-2023. Right of the dashed line = called Silage", x=0.01, ha="left",
             fontsize=10, color=MUTED)
plt.tight_layout(); plt.show()

# Field-years either model gets wrong. The CDL column marks official Grain labels that look like soybeans in the
# USDA Cropland Data Layer (classifier/labels/README.md); no farm record exists for them.
SOY = {(251, 2022), (286, 2022), (230, 2023), (231, 2023), (397, 2023)}
w = fy.loc[te, ["field_id", "farm", "field", "year", "label"] + pt.SMALL].assign(p_baseline=pb, p_prithvi=pp)
w["baseline"] = np.where((pb >= 0.5) == (y[te] == 1), "right", "WRONG")
w["Prithvi"] = np.where((pp >= 0.5) == (y[te] == 1), "right", "WRONG")
w["CDL"] = ["soybeans?" if k in SOY else "" for k in zip(w.field_id, w.year)]
print("Test field-years that the baseline or Prithvi gets wrong:")
display(w[(w.baseline == "WRONG") | (w.Prithvi == "WRONG")].sort_values(["year", "field"]).round(3))

## 10. How much does a prediction depend on which days were clear?

For each test field-year the frames are rebuilt 20 times with scene-days left out at random (probability 0.25) and
scored by the fitted Prithvi model (for a fine-tuned candidate, the first seed). A field-year whose call flips under
this is one where the answer rests on a few images. This is uncertainty per field-year from the imagery alone; it does
not cover label error or a different year's weather. The baseline is not included: its features need the daily
per-field series of `loader.py`, which this notebook does not read.

In [ ]:
tr_main, te_main = SPLITS[MAIN][0]
fitted = CHOSEN["fit"](tr_main, CHOSEN["seeds"][0])
p0 = fitted(te_main)
rng = np.random.default_rng(1)
n_c = CHOSEN["frames"]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    PS = np.stack([fitted(te_main, dropped(te_main, n_c, rng)) for _ in range(N_STAB)])
stab = fy.loc[te_main, ["field_id", "farm", "field", "year", "label"]].assign(
    p=p0.round(3), p_mean=PS.mean(0).round(3), p_sd=PS.std(0).round(3), p_min=PS.min(0).round(3),
    p_max=PS.max(0).round(3), flipped=((PS >= 0.5) != (p0 >= 0.5)).mean(0).round(2))
print(f"{int((stab.flipped > 0).sum())} of {len(stab)} test field-years change their call in at least one of "
      f"{N_STAB} versions; correct per version: {sorted(int(((p >= 0.5) == (y[te_main] == 1)).sum()) for p in PS)}")
display(stab.sort_values(["flipped", "p_sd"], ascending=False).head(12))

## 11. Does the result depend on the least certain training labels?

The chosen setup and the baseline are refitted with the least certain farm-record field-years left out of training
(`pt.EXCLUDE`, see `classifier/labels/README.md`): the 10 Grain field-years whose outline does not fit the field
in that year, then also the 12 Silage labels that rest on a planting record only. The test rows are the same 44 each
time, so the rows compare directly. The candidate is not chosen again.

In [ ]:
misfit = pt.EXCLUDE["outline-misfit"](fy).to_numpy()
plan_silage = pt.EXCLUDE["planting-only-silage"](fy).to_numpy()
variants = {"all 2017-2020 field-years": old, "without outline misfits": old & ~misfit,
            "without outline misfits and planting-only Silage": old & ~misfit & ~plan_silage}
rows = []
for name, keep in variants.items():
    f = [(idx[keep], idx[new])]
    for model, c in (("baseline", BASE), ("Prithvi", CHOSEN)):
        _, p, _ = predict_split(c, f)
        m = metrics(y[idx[new]], p)
        rows.append({"training set": name, "train n": int(keep.sum()), "train Silage": int(y[keep].sum()),
                     "model": model, "correct": m["correct"], "Silage caught": f"{m['TP']}/{m['Silage']}",
                     "Grain called Silage": f"{m['FP']}/{m['Grain']}",
                     "predicted positive rate": m["predicted positive rate"], "ROC-AUC": m["ROC-AUC"],
                     "log loss": m["log loss"]})
SENS = pd.DataFrame(rows)
display(SENS.set_index(["training set", "model"]))

## 12. Every candidate on the headline test (reference only)

Shown so that nothing is hidden, not to pick from: choosing by this table would use the test labels. It shows how
much the result depends on which candidate the 2017-2020 split happened to prefer.

In [ ]:
rows = []
for c in [BASE] + CANDS:
    _, p, _ = (te_main, pb, None) if c is BASE else (te_main, pp, None) if c is CHOSEN else predict_split(c, SPLITS[MAIN])
    m = metrics(y[te_main], p)
    rows.append({"candidate": c["name"] + (" (chosen)" if c is CHOSEN else ""),
                 "choice rank": int(choice.index[choice.candidate == c["name"]][0]) + 1,
                 "choice log loss": float(choice.loc[choice.candidate == c["name"], "log loss"].iloc[0]),
                 "test correct": m["correct"], "Silage caught": f"{m['TP']}/{m['Silage']}",
                 "Grain called Silage": f"{m['FP']}/{m['Grain']}", "test ROC-AUC": m["ROC-AUC"],
                 "test log loss": m["log loss"]})
REF = pd.DataFrame(rows).sort_values("choice rank").reset_index(drop=True)
display(REF)
pr = REF[~REF.candidate.str.startswith(BASE["name"])]
by_frames = {n: pr[pr.candidate.str.contains(f" f{n} ")]["test correct"] for n in FRAME_SETS}
print(f"Prithvi candidates: {pr['test correct'].min()} to {pr['test correct'].max()} of {len(te_main)} correct "
      f"(median {pr['test correct'].median():g}); baseline {int(REF[REF.candidate.str.startswith(BASE['name'])]['test correct'].iloc[0])}.")
print("Median correct by frame set:", {f"{n} frames": float(v.median()) for n, v in by_frames.items()},
      "(descriptive only: the candidates are not independent)")
print(f"total runtime {time.time() - T0:.0f} s")

## 13. Earlier Prithvi results in this workspace, if present

`results/evaluate/summary.md` (frozen Prithvi 300M scored by the team's shared evaluator, which also has a
train-2017-20, test-2021-23 split) and `results/prithvi/summary.md` (the mixed-year run) exist only on the GPU
workspace. They are shown here for comparison when found.

In [ ]:
for rel in ("results/evaluate/summary.md", "results/prithvi/summary.md"):
    f = PRITHVI / rel
    if f.exists():
        display(Markdown(f"---\n**`classifier/prithvi/{rel}`**\n\n" + f.read_text(encoding="utf-8")))
    else:
        print(f"classifier/prithvi/{rel}: not in this workspace")

## 14. Save

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
choice.to_csv(OUT / "choice_scores.csv", index=False)
PRED.to_csv(OUT / "test_predictions.csv", index=False)
MET.to_csv(OUT / "metrics.csv", index=False)
REF.to_csv(OUT / "all_candidates_reference_only.csv", index=False)
stab.to_csv(OUT / "date_dropout_stability.csv", index=False)
SENS.to_csv(OUT / "label_sensitivity.csv", index=False)

run = {"finished_utc": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"), "runtime_s": round(time.time() - T0),
       "chosen": {k: CHOSEN.get(k) for k in ("name", "kind", "size", "frames", "aug", "epochs", "inputs")},
       "cuts_chosen_on_2017_2020": cut,
       "settings": dict(SIZES=SIZES, SCALE=SCALE, FRAME_SETS=FRAME_SETS, KS=KS, C_HEAD=C_HEAD, FT_SIZE=FT_SIZE,
                        RUN_FINETUNE=RUN_FINETUNE, EPOCHS=EPOCHS, SEEDS=SEEDS, P_DROP=P_DROP, N_AUG=N_AUG, N_STAB=N_STAB),
       "frames": {str(n): INP[n]["windows"] for n in FRAME_SETS},
       "models": {f"{s} f{n}": info for (s, n), (_, info) in ENC.items()},
       "inputs_sha256": {"field_years.csv": pt.sha256(PRITHVI / "field_years.csv"),
                         **{f"{v}.npz": pt.sha256(CHIPS / f"{v}.npz") for v in years}},
       "versions": {"python": platform.python_version(), "torch": torch.__version__, "device": DEVICE,
                    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
                    "sklearn": __import__("sklearn").__version__, "timm": __import__("timm").__version__}}
(OUT / "run.json").write_text(json.dumps(run, indent=1, default=str))

m5 = MET[(MET.split == MAIN) & (MET.cut == 0.5)].set_index("model")
keep = ["correct", "accuracy 95%", "predicted positive rate", "actual positive rate", "Silage recall (TPR)",
        "Silage recall 95%", "Silage precision", "false positive rate", "Grain recall", "Grain precision", "ROC-AUC"]
lines = ["# Prithvi-EO-2.0 with a year-based split: results", "",
         "Generated by `prithvi_year_split.ipynb`. The candidate is chosen on hold-out-one-year inside 2017-2020 "
         "(lowest pooled log loss); the test is scored once for it and for the 5-feature logistic baseline fitted on "
         "the same rows. Counts use a 0.5 cut.", "",
         f"**Chosen: {CHOSEN['name']}**" + (f", {CHOSEN['epochs']} epochs, mean of {len(SEEDS)} seeds." if
                                           CHOSEN["kind"] == "finetune" else "."), "",
         f"## {MAIN} ({len(te_main)} field-years, {int(y[te_main].sum())} Silage)", "",
         pt.md_table(m5[keep].T.reset_index().rename(columns={"index": "metric"})), "",
         f"The baseline is wrong on {errors} of {len(te_main)}; a perfect model would win {errors}-0 "
         f"(McNemar p = {best_p:.4f}), so this test cannot show a significant improvement over the baseline.", "",
         "## Every year-based split", "", pt.md_table(MC), "",
         "## Candidates on the 2017-2020 choice split (top 10)", "", pt.md_table(choice.head(10)), "",
         "## Training without the least certain labels (same 44 test field-years)", "", pt.md_table(SENS), "",
         f"## Date dropout on the test field-years ({N_STAB} versions, p = {P_DROP})", "",
         f"{int((stab.flipped > 0).sum())} of {len(stab)} test field-years change their call in at least one version.",
         ""]
(OUT / "summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("wrote", sorted(p.name for p in OUT.iterdir()), "to", OUT)

## 15. Limits

- **One farm, one management team.** All labels are from BARC, Maryland. Nothing here tests another region. The
  Central Farm split is only a partial spatial holdout.
- **Small test.** 44 field-years, 12 Silage: one Silage error moves recall by 8 points, and no model can be shown to
  beat the baseline on it (section 9). Differences of one or two field-years are not findings.
- **Training labels.** 56 of the 83 rest on a planting record only, 12 of them Silage; section 11 shows what changes
  without them. 5 official Grain labels look like soybeans in the Cropland Data Layer (marked in section 9).
- **Model choice is still weakly informed.** 20 candidates on 83 field-years. Section 12 shows the spread across
  candidates on the test.
- **Imagery.** Prithvi was pretrained on NASA HLS; these frames are built from Sentinel-2 L2A and Landsat without
  HLS's BRDF and bandpass adjustments. Long cloud gaps (for example 2022, mid-August to mid-September) leave some
  windows without a clear reading; those frames are filled from the nearest window.
- **Not used.** Sentinel-1 radar, weather, and a corn mask from the Cropland Data Layer. Without a corn mask the model
  cannot be applied to the unlabeled field-years, most of which are not corn.
- **Date dropout** makes training less sensitive to which days were clear. It adds no labels and no field-years.